# Day 1: Load the H&M data and choose a time window

Plan for today:
1. Look at the column names (without loading the data)
2. Decide which columns to keep
3. Load transactions with small dtypes to save memory
4. Count things and choose how many recent weeks to keep
5. Save the result for Day 2

In [1]:
import pandas as pd
import pyarrow.parquet as pq

RAW_FOLDER = '../data/RAW'
PROCESSED_FOLDER = '../data/processed'

## Step 1: Look at the columns only
`pq.read_schema` reads just the column names and types, not the rows, so it is instant.

In [2]:
print(pq.read_schema(RAW_FOLDER + '/transactions_train.parquet'))

t_dat: string
customer_id: string
article_id: int64
price: double
sales_channel_id: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 854


In [3]:
print(pq.read_schema(RAW_FOLDER + '/customers.parquet'))

customer_id: string
FN: double
Active: double
club_member_status: string
fashion_news_frequency: string
age: double
postal_code: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1113


In [4]:
print(pq.read_schema(RAW_FOLDER + '/articles.parquet'))

article_id: int64
product_code: int64
prod_name: string
product_type_no: int64
product_type_name: string
product_group_name: string
graphical_appearance_no: int64
graphical_appearance_name: string
colour_group_code: int64
colour_group_name: string
perceived_colour_value_id: int64
perceived_colour_value_name: string
perceived_colour_master_id: int64
perceived_colour_master_name: string
department_no: int64
department_name: string
index_code: string
index_name: string
index_group_no: int64
index_group_name: string
section_no: int64
section_name: string
garment_group_no: int64
garment_group_name: string
detail_desc: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 3558


## Step 2: Which columns I keep, and why

**Transactions: keep all 5** (`t_dat, customer_id, article_id, price, sales_channel_id`)
- `price` and `sales_channel_id` are cheap to keep (4 bytes and 1 byte per row).
- ⚠️ They describe a *transaction*, not an *article*. When I recommend an article, nobody has bought it yet, so there is no price paid or channel. They are **not features as they are**. Later I can turn price into an article feature, e.g. *median price of the article in the training weeks*.

**Customers: keep** `customer_id, age, club_member_status, fashion_news_frequency, FN, Active`
- **Drop `postal_code`.** It has ~350k different values. Together with age it could identify a real person, which is a privacy (GDPR) risk a retailer would not ship.

**Articles: keep** `article_id, product_code` and the readable category names (product type, group, colour, department, section, garment group, index group)
- Drop the number columns like `product_type_no`. Each one is just a duplicate of its `_name` column.
- Drop `prod_name`. It is almost the same as `product_code`.
- Keep `product_code`. Several articles share one product code (same garment, different colours). Later I'll check that the model doesn't just recommend 5 colours of the same T-shirt.
- Drop `detail_desc` (free text) for V1. Using it would turn this into an NLP project.

## Step 3a: Load transactions the normal way (to see the memory cost)

In [5]:
transactions = pd.read_parquet(RAW_FOLDER + '/transactions_train.parquet')

print('Rows:', len(transactions))
print(transactions.dtypes)

memory_before = transactions.memory_usage(deep=True).sum() / 1024**2
print('Memory before:', round(memory_before), 'MB')

Rows: 31788324
t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object
Memory before: 3456 MB


In [6]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## Step 3b: Make the dtypes smaller

**customer_id** is a 64-character text. Storing 31 million of those is expensive.
I give every customer a number instead: their row position in the customers file.
- I use the **customers file** as the master list, so every table uses the same numbers and joins always match.
- I name it `customer_join_idx` because it is **only for joining tables**. The model will get its own customer vocabulary on Day 4, built from the training weeks only, to avoid leakage.
- Why not `category` dtype? Each table would number its categories on its own, so the same customer could get different codes in different tables.
- Why int32? 1.4M customers fits easily, and it takes half the memory of int64.

In [7]:
customers = pd.read_parquet(RAW_FOLDER + '/customers.parquet', columns=['customer_id'])

# Make the lookup table: customer_join_idx 0, 1, 2, ... next to each customer_id
customers['customer_join_idx'] = range(len(customers))
customers.head()

,customer_id,customer_join_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4


In [8]:
# A dictionary: {'00000dba...': 0, '0000423b...': 1, ...}
id_to_number = dict(zip(customers['customer_id'], customers['customer_join_idx']))

# Replace each text id with its number (takes a few seconds on 31M rows)
transactions['customer_id'] = transactions['customer_id'].map(id_to_number)

# Check the join: a customer not in the customers file would become NaN here
print('Customers missing from customers file:', transactions['customer_id'].isna().sum())

Customers missing from customers file: 0


**article_id** is already a number in both files. The leading zero only exists in the image folder names, so I keep it as a number everywhere: int32 (the largest id is ~959 million, which fits).

**t_dat** must be a real date, not text, because the train/validation/test split depends on it.

In [9]:
transactions = transactions.rename(columns={'customer_id': 'customer_join_idx'})

transactions['customer_join_idx'] = transactions['customer_join_idx'].astype('int32')
transactions['article_id'] = transactions['article_id'].astype('int32')
transactions['price'] = transactions['price'].astype('float32')
transactions['sales_channel_id'] = transactions['sales_channel_id'].astype('int8')
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print(transactions.dtypes)

memory_after = transactions.memory_usage(deep=True).sum() / 1024**2
print('Memory after:', round(memory_after), 'MB')
print('Saved:', round(memory_before / memory_after, 1), 'times smaller')

t_dat                datetime64[us]
customer_join_idx             int32
article_id                    int32
price                       float32
sales_channel_id               int8
dtype: object
Memory after: 637 MB
Saved: 5.4 times smaller


In [10]:
# Check the other join: every article in transactions should exist in the articles file
articles = pd.read_parquet(RAW_FOLDER + '/articles.parquet', columns=['article_id'])
missing_articles = ~transactions['article_id'].isin(articles['article_id'])
print('Articles missing from articles file:', missing_articles.sum())

Articles missing from articles file: 0


## Step 4a: Basic counts

In [11]:
print('First date:', transactions['t_dat'].min())
print('Last date: ', transactions['t_dat'].max())
print('Rows:            ', len(transactions))
print('Unique customers:', transactions['customer_join_idx'].nunique())
print('Unique articles: ', transactions['article_id'].nunique())

First date: 2018-09-20 00:00:00


Last date:  2020-09-22 00:00:00
Rows:             31788324


Unique customers: 1362281


Unique articles:  104547


## Step 4b: Look at the most recent weeks
Kaggle's test week runs Wednesday to Tuesday, so I group the dates into weeks that **end on Tuesday** (`'W-TUE'`).

In [12]:
weekly = transactions.groupby(pd.Grouper(key='t_dat', freq='W-TUE')).agg(
    rows=('article_id', 'size'),
    customers=('customer_join_idx', 'nunique'),
    articles=('article_id', 'nunique'),
)
weekly.tail(16)

,rows,customers,articles
t_dat,,,
2020-06-09,310996,78193,21894
2020-06-16,298890,77306,21785
2020-06-23,549443,120190,23281
2020-06-30,517428,115992,22837
2020-07-07,337419,85648,21503
2020-07-14,286672,75996,20598
2020-07-21,289528,77277,20553
2020-07-28,299469,80591,21204
2020-08-04,319469,82802,20646


The weeks ending 2020-06-23 and 2020-06-30 have about **1.8x** the normal rows. That's a summer clearance sale, not normal behaviour.

## Step 4c: Compare a few window sizes

In [13]:
last_date = transactions['t_dat'].max()

for weeks in [4, 8, 12, 16, 26, 52]:
    start_date = last_date - pd.Timedelta(weeks=weeks)
    window = transactions[transactions['t_dat'] > start_date]
    print(weeks, 'weeks:',
          len(window), 'rows,',
          window['customer_join_idx'].nunique(), 'customers,',
          window['article_id'].nunique(), 'articles')

4 weeks: 1043390 rows, 233174 customers, 28297 articles


8 weeks: 2173875 rows, 378688 customers, 35576 articles


12 weeks: 3386963 rows, 489553 customers, 40772 articles


16 weeks: 5063720 rows, 594161 customers, 45774 articles


26 weeks: 8092617 rows, 744034 customers, 51316 articles


52 weeks: 14827645 rows, 992532 customers, 70768 articles


## Step 5: My choice is the last 12 weeks (2020-07-01 to 2020-09-22)

- **Room for the split:** 10 weeks train + 1 week validation (09-09 to 09-15) + 1 week test (09-16 to 09-22).
- **Season:** summer into early autumn, the same season as the test week.
- **Skips the clearance sale:** it starts the day after that spike ends.
- **Fresh articles:** ~41k articles still selling, vs ~105k in the full catalogue.
- **Hardware:** ~3.4M rows is small enough for fast training on a 16 GB laptop.

⚠️ 12 weeks is a **hypothesis**, not a final answer. **Day 9 experiment:** compare 12 vs 16 weeks on the validation week and keep whichever scores better.

In [14]:
start_date = last_date - pd.Timedelta(weeks=12)
transactions_12w = transactions[transactions['t_dat'] > start_date]

print('From', transactions_12w['t_dat'].min(), 'to', transactions_12w['t_dat'].max())
print('Rows:', len(transactions_12w))
print('Memory:', round(transactions_12w.memory_usage(deep=True).sum() / 1024**2), 'MB')

From 2020-07-01 00:00:00 to 2020-09-22 00:00:00
Rows: 3386963
Memory: 68 MB


In [15]:
import os
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

transactions_12w.to_parquet(PROCESSED_FOLDER + '/transactions_12w.parquet', index=False)

# Save the lookup table so later notebooks can turn numbers back into customer ids
customers.to_parquet(PROCESSED_FOLDER + '/customer_join_idx.parquet', index=False)

print('Saved!')

Saved!
